## Deliverable 2

task 1:

In [ ]:
import re
from pathlib import Path


def read_file(file_path: str | Path) -> tuple[str, str]:
    """Read the file and returns a tuple with the content and the encoding like (content, encoding)"""
    path = Path(file_path)
    try:
        file_text_content = path.read_text(encoding="utf-8")
        return file_text_content, "utf-8"
    except UnicodeDecodeError:
        file_text_content = path.read_text(encoding="iso-8859-1")
        return file_text_content, "iso-8859-1"


def parse_log(log_text: str):
    """Parse log text into a list of record dictionaries."""
    pattern = re.compile(
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})] "
        r"\[(?P<register_id>[^\]]+)\] "
        r"item=(?P<item>\w+) "
        r"price=(?P<price>\d+\.\d+) "
        r"(?P<currency>[A-Z]{3}) "
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
    )
    records = []
    for line in log_text.splitlines():
        match = pattern.search(line)
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

Task 2: 


In [ ]:
import json


def find_log_files(dir_path: str | Path) -> list[Path]:
    """Find all log files recursively in a directory and return a list of pathlib.Path objects."""
    dir_path = Path(dir_path)
    log_files: list[Path] = []

    for item in dir_path.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)
    return log_files


def find_active_registers(data) -> set:
    """Find all registerd ids where "active" is True."""
    active_registers = set()

    if isinstance(data, dict):
        if "id" in data and data.get("active") is True:
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_active_registers(item))
    return active_registers


with open("data/store_config.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

active_registers_in_metadata = find_active_registers(metadata)


Task 3: 

In [ ]:
import numpy as np


def removing_price_outliers(data) -> np.ndarray:
    """Remove price outliers like 999.0 and return a 1D numpy array."""
    price = np.array(data, dtype=float).ravel()
    mean = np.mean(price)
    std = np.std(price)
    outliers = np.abs(price - mean) > 3 * std

    valid_prices = price[~outliers]
    valid_median = np.median(valid_prices)

    price[outliers] = valid_median
    return price


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Compute the refund ratio."""
    refund_ratio = refund_count / (sales_count + 10**-6)
    return refund_ratio

Task 4: 

In [ ]:
import pandas


def convert_transaction_dicts_df(records) -> pandas.DataFrame:
    """Convert a list of transactions to a pandas DataFrame, convert "timestamp" to 
    datetime, and sort the DataFrame in chronological order.
    """
    df = pandas.DataFrame(records)

    if "timestamp" in df.columns:
        df["timestamp"] = pandas.to_datetime(df["timestamp"])
        df = df.sort_values(by="timestamp", ascending=True, ignore_index=True)
    return df


def total_price_per_hour(df: pandas.DataFrame) -> pandas.DataFrame:
    """Calculate the total sum per hour, per item."""
    per_hour = df.groupby(["item", pandas.Grouper(key="timestamp", freq="1h")])["price"].sum().reset_index()
    return per_hour


def create_pivot_table(df: pandas.DataFrame) -> pandas.DataFrame:
    """Create a pivot table that displays total transaction revenue grouped by item."""
    pivot_table = df.pivot_table(
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0,
    )
    return pivot_table


def refund_per_item(df: pandas.DataFrame) -> pandas.Series:
    """Calculate the refund ratio per item."""
    counts = pandas.crosstab(df["item"], df["status"])
    sales = counts["SUCCESS"].to_numpy()
    refunds = counts["REFUND"].to_numpy()
    ratios = compute_refund_ratio(sales, refunds)
    return pandas.Series(ratios, index=counts.index, name="refund_ratio")

Task 5: 

In [ ]:
import csv


def extracting_active_registers(data_path = Path("data")) -> tuple[set, list]:
    """Find all log files and extract active register IDs."""
    log_files = find_log_files(data_path)
    json_path = data_path / "store_config.json"

    with open(json_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
        
    active_registers = find_active_registers(metadata)
    return active_registers, log_files


def ingest_txtlogs_parse_records(log_files: list[Path]) -> list:
    """Ingest text logs and parse records into a list of dictionaries."""
    all_records = []

    for file_path in log_files:
        content, encoding = read_file(file_path)
        records = parse_log(content)
        all_records.extend(records)
    return all_records

    
def create_initial_dataframe(log_files: list[Path]) -> pandas.DataFrame:
    """Create initial dataframe."""
    records = ingest_txtlogs_parse_records(log_files)
    initial_df = convert_transaction_dicts_df(records)
    return initial_df

    
def clean_outliers_df(df: pandas.DataFrame) -> pandas.DataFrame:
    """Clean outliers."""
    cleaned_df = df.copy()
    cleaned_prices = removing_price_outliers(cleaned_df["price"])
    cleaned_df["price"] = cleaned_prices
    return cleaned_df


def currency_correction(df: pandas.DataFrame) -> pandas.DataFrame:
    """Correct currency, if the currency is "NOK" multiply price with 10.0."""
    cleaned_df = df.copy()
    mask = cleaned_df["currency"] == "NOK"
    cleaned_df.loc[mask, "price"] = cleaned_df.loc[mask, "price"] * 10.0
    return cleaned_df


def generate_reports(cleaned_df: pandas.DataFrame):
    """Generate reports/output files for cleaned transactions, hourly revenue, 
    status pivot table and item refund ratios."""
    cleaned_df.to_csv("cleaned_transactions.csv", index=False)
    
    hourly_df = (
        cleaned_df.resample("1h", on="timestamp")["price"].sum().reset_index()
    )
    hourly_df.to_json("hourly_revenue.json", indent=4)

    pivot_df = create_pivot_table(cleaned_df)
    pivot_df.to_csv("status_pivot_table.csv")

    refund_ratios = refund_per_item(cleaned_df)
    refund_ratios.to_csv("item_refund_ratios.csv")


def workflow():
    """Complete workflow."""
    active_registers, log_files = extracting_active_registers()
    initial_df = create_initial_dataframe(log_files)
    df_no_outliers = clean_outliers_df(initial_df)
    cleaned_df = currency_correction(df_no_outliers)

    generate_reports(cleaned_df)

workflow()

Git hub repository link: https://github.com/victoriafuru1/Deliverable2